# 06 | Decision thresholds and holdout evaluation

F1-based threshold selection, final holdout evaluation, and balanced-accuracy threshold tuning.

**Source:** `archive/Project_fin_original.ipynb`. The original computations, structure, and results have been preserved. The notebook text has been translated into English; standalone setup and local checkpoints support reproducible execution.


In [4]:
# Standalone setup: no src/ folder required.
# Run the notebooks in numerical order (02 -> 07); 01 is optional exploration.
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

working_dir = Path.cwd().resolve()
# Works when VS Code/Jupyter starts in the repository root or notebooks/.
if working_dir.name == "notebooks":
    project_root = working_dir.parent
elif (working_dir / "notebooks").is_dir():
    project_root = working_dir
elif (working_dir.parent / "notebooks").is_dir():
    project_root = working_dir.parent
else:
    project_root = working_dir

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

HORIZON = pd.Timedelta(days=10)
CUTOFFS = pd.date_range(start="2018-10-11", end="2018-11-10", freq="10D")
FEATURES_12 = [
    "n_sessions_10d", "n_active_days_10d", "n_songs_10d",
    "n_likes_10d", "n_dislikes_10d", "n_errors_10d",
    "n_playlist_adds_10d", "total_song_length_10d", "n_artists_10d",
    "downgraded_10d", "is_paid", "songs_trend_10d"
]
EVENT_COLS = [
    "userId", "time", "registration", "page", "sessionId", "level",
    "gender", "length", "artist", "song", "status", "method",
    "userAgent", "auth", "firstName", "lastName", "itemInSession", "location"
]

def raw_path(kind):
    if kind not in ("train", "test"):
        raise ValueError("kind must be 'train' or 'test'")
    filename = f"{kind}.parquet"
    candidates = [
        project_root / filename,
        project_root / "data" / "raw" / filename,
        project_root / "notebooks" / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Missing {filename}. Put it in the repository root or data/raw/. "
        f"Current root: {project_root}"
    )

def load_events(kind):
    frame = pd.read_parquet(raw_path(kind), columns=EVENT_COLS)
    frame["userId"] = frame["userId"].astype(str)
    frame["time"] = pd.to_datetime(frame["time"])
    frame["registration"] = pd.to_datetime(frame["registration"])
    return frame

def data_path(filename):
    folder = project_root / "data" / "processed"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def artifact_path(filename):
    folder = project_root / "artifacts"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def save_cv_state(dev_data, holdout_data, cv_splits):
    dev_data.to_parquet(data_path("dev_data.parquet"), index=False)
    holdout_data.to_parquet(data_path("holdout_data.parquet"), index=False)
    fold_arrays = {}
    for i, (idx_train, idx_valid) in enumerate(cv_splits):
        fold_arrays[f"train_{i}"] = np.asarray(idx_train, dtype=np.int64)
        fold_arrays[f"valid_{i}"] = np.asarray(idx_valid, dtype=np.int64)
    np.savez_compressed(data_path("cv_splits.npz"), **fold_arrays)

def load_cv_state():
    dev = pd.read_parquet(data_path("dev_data.parquet"))
    holdout = pd.read_parquet(data_path("holdout_data.parquet"))
    with np.load(data_path("cv_splits.npz")) as saved:
        n = sum(name.startswith("train_") for name in saved.files)
        splits = [
            (saved[f"train_{i}"].copy(), saved[f"valid_{i}"].copy())
            for i in range(n)
        ]
    return dev, holdout, splits

print("Project folder:", project_root)

import json
import joblib
dev_data, holdout_data, cv_splits = load_cv_state()
models = joblib.load(artifact_path("model_specs.joblib"))  # uniquement un artefact local de confiance


Project folder: C:\Users\hugov\Documents\M1\Python for data science\Project\Churn_Prediction


In [5]:

from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from sklearn.base import clone

# 1. Obtain out-of-fold predictions
oof_predictions = []

for fold, (train_idx, valid_idx) in enumerate(cv_splits, 1):

    train_fold = dev_data.iloc[train_idx]
    valid_fold = dev_data.iloc[valid_idx]

    model = clone(models["LogisticRegression"])

    model.fit(
        train_fold[FEATURES_12],
        train_fold["churn"]
    )

    proba = model.predict_proba(
        valid_fold[FEATURES_12]
    )[:, 1]

    oof_predictions.append(pd.DataFrame({
        "fold": fold,
        "churn_reel": valid_fold["churn"].to_numpy(),
        "probability": proba
    }))

oof = pd.concat(oof_predictions, ignore_index=True)

# 2. Compute precision and recall at each threshold
precision, recall, thresholds = precision_recall_curve(
    oof["churn_reel"],
    oof["probability"]
)

# Compute F1 score at each threshold
f1_scores = (
    2 * precision[:-1] * recall[:-1] /
    (precision[:-1] + recall[:-1] + 1e-10)
)

# 3. Find the optimal decision threshold
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

# 4. Compare binary predictions
pred_05 = (oof["probability"] >= 0.5).astype(int)
pred_opt = (
    oof["probability"] >= best_threshold
).astype(int)

print("Optimal CV threshold:", round(best_threshold, 4))

for name, predictions in [
    ("Threshold 0.5", pred_05),
    ("Optimized threshold", pred_opt)
]:
    print(f"\n{name}")

    print("Precision:", round(
        precision_score(oof["churn_reel"], predictions,
                        zero_division=0), 4
    ))

    print("Recall:", round(
        recall_score(oof["churn_reel"], predictions,
                     zero_division=0), 4
    ))

    print("F1-score :", round(
        f1_score(oof["churn_reel"], predictions,
                 zero_division=0), 4
    ))

    print("Predicted churns:", predictions.sum())

    print("Confusion matrix:")
    print(confusion_matrix(oof["churn_reel"], predictions))


Optimal CV threshold: 0.0863

Threshold 0.5
Precision: 0.0
Recall: 0.0
F1-score : 0.0
Predicted churns: 0
Confusion matrix:
[[5889    0]
 [ 332    0]]

Optimized threshold
Precision: 0.1312
Recall: 0.2952
F1-score : 0.1816
Predicted churns: 747
Confusion matrix:
[[5240  649]
 [ 234   98]]


In [6]:

from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score,
    f1_score, confusion_matrix
)

# 1. Retrieve the logistic regression model
final_model = clone(models["LogisticRegression"])

# 2. Train on all development data
X_train = dev_data[FEATURES_12]
y_train = dev_data["churn"]

final_model.fit(X_train, y_train)

# 3. Evaluate on the final holdout
X_test = holdout_data[FEATURES_12]
y_test = holdout_data["churn"]

y_proba = final_model.predict_proba(X_test)[:, 1]

# 4. Apply the threshold selected in cross-validation
y_pred = (y_proba >= best_threshold).astype(int)

# 5. Evaluate model performance
print("=== HOLDOUT RESULTS ===")

print("ROC-AUC :", round(
    roc_auc_score(y_test, y_proba), 4
))

print("Average Precision :", round(
    average_precision_score(y_test, y_proba), 4
))

print("Precision:", round(
    precision_score(y_test, y_pred, zero_division=0), 4
))

print("Recall:", round(
    recall_score(y_test, y_pred, zero_division=0), 4
))

print("F1-score :", round(
    f1_score(y_test, y_pred, zero_division=0), 4
))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nActual churners:", y_test.sum())
print("Predicted churners:", y_pred.sum())



=== HOLDOUT RESULTS ===
ROC-AUC : 0.7578
Average Precision : 0.1423
Precision: 0.1643
Recall: 0.4101
F1-score : 0.2346

Confusion matrix:
[[2634  290]
 [  82   57]]

Actual churners: 139
Predicted churners: 347


In [7]:

from sklearn.metrics import balanced_accuracy_score

# Test different classification thresholds
thresholds = np.linspace(0.01, 0.50, 500)

scores = []

for threshold in thresholds:

    predictions = (
        oof["probability"] >= threshold
    ).astype(int)

    score = balanced_accuracy_score(
        oof["churn_reel"],
        predictions
    )

    scores.append(score)

# Find the best threshold
best_idx = np.argmax(scores)

best_threshold_bal = thresholds[best_idx]

print("Best threshold:", round(best_threshold_bal, 4))
print("Balanced Accuracy CV :", round(scores[best_idx], 4))

# Compare with the previous threshold
old_predictions = (
    oof["probability"] >= best_threshold
).astype(int)

old_score = balanced_accuracy_score(
    oof["churn_reel"],
    old_predictions
)

print("Balanced accuracy at previous threshold:", round(old_score, 4))


Best threshold: 0.0503
Balanced Accuracy CV : 0.6407
Balanced accuracy at previous threshold: 0.5925


In [8]:
# Decision thresholds from the original analysis, used for submission
import json
with open(artifact_path("seuils.json"), "w", encoding="utf-8") as f:
    json.dump({"best_threshold": float(best_threshold),
               "best_threshold_bal": float(best_threshold_bal)}, f, indent=2)
print("Thresholds saved.")

Thresholds saved.


In [9]:
from pathlib import Path
import joblib

# Find the project root
project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Define the path to saved models
def artifact_path(filename):
    return project_root / "artifacts" / filename

# Check that the files exist
print("Model found:",
      artifact_path("logreg_gridsearch_best.joblib").exists())

print("Threshold found:",
      artifact_path("logreg_gridsearch_threshold.joblib").exists())


Model found: True
Threshold found: True


In [10]:

from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import joblib

# Load the optimized model and threshold
model_grid = joblib.load(
    artifact_path("logreg_gridsearch_best.joblib")
)

threshold_grid = joblib.load(
    artifact_path("logreg_gridsearch_threshold.joblib")
)

# Holdout predictions
X_holdout = holdout_data[FEATURES_12]
y_holdout = holdout_data["churn"]

scores_grid = model_grid.predict_proba(X_holdout)[:, 1]
pred_grid = (scores_grid >= threshold_grid).astype(int)

# Evaluate
print("GridSearch Holdout Balanced Accuracy:")
print(round(balanced_accuracy_score(y_holdout, pred_grid), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_holdout, pred_grid))


GridSearch Holdout Balanced Accuracy:
0.694

Confusion Matrix:
[[1997  927]
 [  41   98]]


In [11]:

from sklearn.metrics import balanced_accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# 1. Rebuild our original LogisticRegression
baseline = make_pipeline(
    SimpleImputer(
        strategy="constant",
        fill_value=0,
        add_indicator=True,
        keep_empty_features=True
    ),
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        random_state=42
    )
)

# 2. Train on the same development dataset
baseline.fit(
    dev_data[FEATURES_12],
    dev_data["churn"]
)

# 3. Predictions on the same holdout
X_holdout = holdout_data[FEATURES_12]
y_holdout = holdout_data["churn"]

baseline_proba = baseline.predict_proba(X_holdout)[:, 1]
grid_proba = model_grid.predict_proba(X_holdout)[:, 1]

baseline_pred = (baseline_proba >= 0.0503).astype(int)
grid_pred = (grid_proba >= threshold_grid).astype(int)

# 4. Compare Balanced Accuracy
comparison = pd.DataFrame({
    "Model": ["Baseline V3", "GridSearch"],
    "Threshold": [0.0503, threshold_grid],
    "Balanced Accuracy": [
        balanced_accuracy_score(y_holdout, baseline_pred),
        balanced_accuracy_score(y_holdout, grid_pred)
    ]
})

display(comparison.round(4))


,Model,Threshold,Balanced Accuracy
0,Baseline V3,0.0503,0.6932
1,GridSearch,0.0509,0.6940
